In [2]:
from pathlib import Path    
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib import cm
from scipy.interpolate import griddata
from scipy.stats import norm


In [3]:
ts_path = Path.cwd() / "data" / "time_series"

text_path = Path.cwd() / "data" / "text"

images_path = Path.cwd() / "images"
images_path.mkdir(parents=True, exist_ok=True)

csv_files = sorted(ts_path.glob("*.csv"))

In [4]:
print(f"{len(csv_files)} Ticker(s) found in {ts_path}")

ticker_to_df = {}
n_tickers = 100

for i, csv_file in enumerate(csv_files):

    if i >= n_tickers:
        break

    df = pd.read_csv(csv_file)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)
    df = df.dropna(subset=["Date"]).sort_values("Date")
    df["Date"] = df["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    ticker_to_df[csv_file.stem] = df

4213 Ticker(s) found in /home/rodrigodog/latent_fusion/data/time_series


In [7]:
def animate_volatility_surface(surfaces, timestamps, ticker, save_path=None):
    """
    Create 3D animation of volatility surface evolution.
    """
    
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    
    def update(frame):
        ax.clear()
        
        vols_data = surfaces[frame]
        vols_df = pd.DataFrame(vols_data)
        
        K = vols_df["K"].values
        T = vols_df["T"].values
        IV = vols_df["IV"].values
        
        if len(K) > 3:
            try:
                xi = np.linspace(K.min(), K.max(), 20)
                ti = np.linspace(T.min(), T.max(), 15)
                Xi, Ti = np.meshgrid(xi, ti)
                Zi = griddata((K, T), IV, (Xi, Ti), method='cubic')
                
                surf = ax.plot_surface(Xi, Ti, Zi, cmap=cm.coolwarm, alpha=0.8, edgecolor='none')
                
                ax.scatter(K, T, IV, color='black', s=30, alpha=0.6)
            except:
                ax.scatter(K, T, IV, c=IV, cmap=cm.coolwarm, s=50)
        
        ax.set_xlabel('Strike (K)')
        ax.set_ylabel('Maturity (days)')
        ax.set_zlabel('Implied Vol')
        ax.set_zlim(0, IV.max() * 1.2 if len(IV) > 0 else 1)
        ax.set_title(f'{ticker} Volatility Surface\n{timestamps[frame].strftime("%Y-%m-%d")}')
        ax.view_init(elev=25, azim=45)
    
    ani = FuncAnimation(fig, update, frames=len(surfaces), interval=200, repeat=True)
    
    if save_path:
        save_path = save_path.replace('.mp4', '.gif')
        print(f"Saving animation to {save_path}...")
        ani.save(save_path, writer='pillow', fps=5)
    
    plt.tight_layout()
    return ani

In [8]:
videos_path = Path.cwd() / "videos"
videos_path.mkdir(parents=True, exist_ok=True)

select_tickers = list(ticker_to_df.keys())[:1]

for ticker in select_tickers:
    df = ticker_to_df[ticker].tail(1000).reset_index(drop=True)
    
    print(f"\n{'='*60}")
    print(f"Building volatility surface for {ticker.upper()}")
    print(f"{'='*60}")
    
    surfaces, timestamps, features = compute_realized_vol_surface(
        df,
        window=30,
        strikes_range=(0.85, 1.15),
        maturities=(5, 10, 20, 60)
    )
    
    if len(surfaces) > 0:
        print(f"✓ Generated {len(surfaces)} surface snapshots")
        print(f"  Date range: {timestamps[0].strftime('%Y-%m-%d')} to {timestamps[-1].strftime('%Y-%m-%d')}")
        print(f"  Features shape: {features.shape}")
        
        video_path = videos_path / f"{ticker.upper()}_volatility_surface.gif"
        ani = animate_volatility_surface(surfaces, timestamps, ticker.upper(), save_path=str(video_path))
        print(f"✓ Video saved to {video_path}")
        plt.close()
        
        fig = plot_surface_features(features, ticker.upper())
        fig_path = videos_path / f"{ticker.upper()}_surface_features.png"
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"✓ Features plot saved to {fig_path}")
        plt.close()
        
        print(f"\nSurface Feature Statistics for {ticker.upper()}:")
        print(features[["ATM_Vol", "Term_Slope", "Skew", "Vol_of_Vol"]].describe())
    else:
        print(f"✗ Could not generate surfaces for {ticker}")


Building volatility surface for A
✓ Generated 910 surface snapshots
  Date range: 2021-05-19 to 2024-12-30
  Features shape: (910, 5)
Saving animation to /home/rodrigodog/latent_fusion/videos/A_volatility_surface.gif...
✓ Video saved to /home/rodrigodog/latent_fusion/videos/A_volatility_surface.gif
✓ Features plot saved to /home/rodrigodog/latent_fusion/videos/A_surface_features.png

Surface Feature Statistics for A:
          ATM_Vol  Term_Slope   Skew  Vol_of_Vol
count  910.000000  910.000000  910.0  910.000000
mean     0.250728    0.037742    0.0    0.049058
std      0.075030    0.069976    0.0    0.026841
min      0.084122   -0.252360    0.0    0.003603
25%      0.200860    0.000385    0.0    0.028361
50%      0.239889    0.041239    0.0    0.043940
75%      0.298995    0.080650    0.0    0.064760
max      0.492483    0.253809    0.0    0.161554
